# 07. Lectura de concordancias: los ojos en Dickens

En los cuadernos anteriores contamos palabras, hicimos tablas y comparamos vectores. Ahora vamos a **leer**. Pero no un texto: miles de fragmentos a la vez.

Una **concordancia** reúne todas las apariciones de una palabra (el **nodo**) con un poco de contexto a cada lado. Se suele mostrar en formato **KWIC** (*Key Word In Context*), con el nodo alineado al centro:

```
          ... he turned his | eyes | upon the fire, and ...
   ... she raised her tearful | eyes | to his face ...
```

Leer concordancias es el método más antiguo de la lingüística de corpus (Sinclair, 1991). También es una forma de **lectura distante** (Moretti, 2013): en lugar de leer una novela de principio a fin, vemos cómo se comporta una palabra en quince novelas a la vez.

En este cuaderno vamos a:

1. Obtener la concordancia de *eyes* en las novelas de Charles Dickens.
2. Ordenarla, agruparla y filtrarla para encontrar patrones.
3. Ver cómo Dickens usa los ojos de sus personajes como **lenguaje corporal**, sobre todo en las *suspensiones*.
4. Probar una agrupación automática por significado.

**Por qué *eyes* y Dickens.** Michaela Mahlberg (2013) mostró que en Dickens los grupos de palabras como *his eyes fixed on* o *with his hands in his pockets* caracterizan a los personajes. Este ejemplo se ha vuelto un clásico de la estilística de corpus.

**Créditos.** Este cuaderno adapta los materiales del curso *Reading Concordances* de Stephanie Evert y su equipo (ESSLLI 2025), en particular `flexiconc_introduction.ipynb` y `flexiconc_import.ipynb`. Los materiales originales están en [github.com/reading-concordances/teaching](https://github.com/reading-concordances/teaching).

## 0. Preparación

Usaremos **FlexiConc**, una biblioteca de Python para organizar y leer concordancias. Las concordancias vienen de **CLiC** ([clic-fiction.com](https://clic-fiction.com)), un servidor público con novelas inglesas del siglo XIX. No hace falta cuenta ni contraseña.

La celda de abajo instala todo en **Google Colab**. Tiene dos pasos:

1. `apt-get` instala la biblioteca ICU del sistema. FlexiConc la necesita para **ordenar** alfabéticamente según las reglas de cada idioma. Sin ella, la instalación de `pyicu` falla.
2. `pip` instala FlexiConc con sus extras: `notebooks` (los controles interactivos), `web` (la conexión con CLiC), `ICU` (el orden alfabético), `partitioning` y `annotation` (agrupación y embeddings).

Tarda unos minutos. Si Colab te pide **reiniciar la sesión** al terminar, hazlo y continúa con la celda siguiente.

> **En tu computadora.** Instala ICU antes de `pip`. En Ubuntu: `sudo apt install libicu-dev pkg-config`. En macOS: `brew install icu4c pkg-config`. En Windows lo más sencillo es usar `conda install -c conda-forge pyicu` antes de instalar FlexiConc. Si no quieres lidiar con esto, usa Colab.

In [ ]:
!apt-get -qq install -y libicu-dev pkg-config > /dev/null
!pip install -q 'flexiconc[notebooks,web,ICU,partitioning,annotation]'

Importamos la clase `Concordance` y cuatro funciones auxiliares para cuadernos:

- `show_kwic()` muestra una concordancia en formato KWIC.
- `show_analysis_tree()` muestra el **árbol de análisis** (lo explicamos en la sección 3).
- `add_node_ui()` y `add_annotation_ui()` abren menús interactivos para aplicar algoritmos sin escribir código.

In [ ]:
from flexiconc import Concordance
from flexiconc.utils.notebook_utils import add_node_ui, add_annotation_ui, show_kwic, show_analysis_tree

---
## 1. La concordancia de *eyes* en Dickens

CLiC tiene varios corpus. Usaremos `DNov`, que contiene las 15 novelas de Dickens. Cada novela tiene una clave de dos o tres letras:

| Clave | Novela | Clave | Novela |
|---|---|---|---|
| PP | *The Pickwick Papers* (1836) | DS | *Dombey and Son* (1846) |
| OT | *Oliver Twist* (1837) | DC | *David Copperfield* (1849) |
| NN | *Nicholas Nickleby* (1838) | BH | *Bleak House* (1852) |
| OCS | *The Old Curiosity Shop* (1840) | HT | *Hard Times* (1854) |
| BR | *Barnaby Rudge* (1841) | LD | *Little Dorrit* (1855) |
| MC | *Martin Chuzzlewit* (1843) | TTC | *A Tale of Two Cities* (1859) |
| GE | *Great Expectations* (1860) | OMF | *Our Mutual Friend* (1864) |
| ED | *The Mystery of Edwin Drood* (1870) | | |

`retrieve_from_clic()` pide la concordancia al servidor. `query` es una lista de palabras o frases. Con `add_clic_subsets_to_metadata=True` pedimos que cada línea diga en qué parte del texto aparece. Lo usaremos en la sección 6.

In [ ]:
C = Concordance()
C.retrieve_from_clic(query=["eyes"], corpora=["corpus:DNov"], add_clic_subsets_to_metadata=True)
C.root.line_count

Son más de 4000 líneas. Nadie las va a leer todas, y no hace falta.

El objeto `C` guarda dos tablas de pandas, como las del cuaderno 04:

- `C.metadata` tiene **una fila por línea** de la concordancia: de qué novela viene (`text_id`), capítulo, párrafo, oración y en qué parte del texto aparece (`clic_subset`).
- `C.tokens` tiene **una fila por palabra**. La columna `offset` dice la posición respecto al nodo: `0` es *eyes*, `-1` es la palabra inmediatamente a la izquierda, `1` la de la derecha, etcétera.

En lingüística de corpus esas posiciones se nombran **L1, L2, …** (izquierda) y **R1, R2, …** (derecha). En FlexiConc, L1 es `-1` y R1 es `1`.

In [ ]:
C.metadata.head()

In [ ]:
C.tokens.head(25)

¿Qué novelas tienen más ojos? Como `metadata` es un DataFrame, podemos usar `value_counts()`:

In [ ]:
C.metadata["text_id"].value_counts()

**Ojo con estas cifras.** Son frecuencias absolutas. *Dombey and Son* es mucho más larga que *Hard Times*, así que es natural que tenga más ojos. Para comparar novelas habría que normalizar (por ejemplo, apariciones por cada 10 000 palabras), como hicimos con las variantes en el cuaderno 04.

Ahora veamos las primeras diez líneas en formato KWIC. Con `metadata_columns` agregamos columnas de la tabla de metadatos.

In [ ]:
show_kwic(C.root, n=10, metadata_columns=("text_id", "chapter"))

---
## 2. Vistas de la concordancia y el árbol de análisis

Leer concordancias es un proceso: tomamos una muestra, ordenamos, agrupamos, filtramos, volvemos atrás. FlexiConc registra cada paso como un **nodo** en un **árbol de análisis**. La raíz (`C.root`) es la concordancia completa.

Hay dos tipos de nodos:

- **Nodos de subconjunto** (*subset nodes*): se quedan con **algunas** líneas. Se crean con un algoritmo de selección, por ejemplo una muestra aleatoria.
- **Nodos de arreglo** (*arrangement nodes*): conservan **todas** las líneas pero cambian cómo se presentan. Se crean con algoritmos de ordenamiento, de *ranking* o de agrupación.

Cada nodo cuelga de otro. Así podemos regresar a cualquier punto y probar otro camino sin perder lo que hicimos.

### Una muestra aleatoria

El primer paso con una concordancia grande suele ser leer una **muestra aleatoria** para darse una idea general. La semilla (`seed`) hace que la muestra sea reproducible: con la misma semilla salen las mismas líneas.

In [ ]:
muestra = C.root.add_subset_node(("Random Sample", {"sample_size": 20, "seed": 42}))
show_kwic(muestra, metadata_columns=("text_id", "chapter"))

### La alternativa interactiva

Para escribir esa línea hay que saber el nombre exacto del algoritmo y de sus parámetros. `add_node_ui()` muestra menús para elegirlos.

Prueba: elige *subset*, luego *Random Sample*, cambia el tamaño o la semilla y haz clic en **OK**. Debajo del botón aparece el código que FlexiConc ejecutó. Es el mismo tipo de línea que escribimos arriba.

**Para un análisis reproducible**, copia ese código a una celda normal. Si vuelves a abrir el cuaderno, los menús no recuerdan lo que elegiste, pero el código sí.

In [ ]:
ui = add_node_ui(C.root)

Cada clic en **OK** agrega un nodo nuevo al árbol. Los números entre corchetes son los identificadores de cada nodo.

In [ ]:
show_analysis_tree(C)

Si probaste varias muestras con los menús, puedes quitar los nodos que sobran. `find_node_by_id()` localiza un nodo y `remove()` lo elimina con todo lo que cuelga de él. Cambia el número por el de un nodo que quieras borrar (el 1 es nuestra `muestra`, así que no lo borres).

In [ ]:
# nodo = C.find_node_by_id(2)
# nodo.remove()
show_analysis_tree(C)

---
## 3. Ordenar

La forma clásica de leer una concordancia es **ordenarla** por el contexto. Al ordenar por la izquierda, las líneas con la misma palabra antes del nodo quedan juntas y los patrones saltan a la vista.

Ordenar es un nodo de **arreglo**. Lo colgamos de la raíz, no de la muestra, porque queremos ver todas las líneas.

- `sorting_scope="left"` ordena por todo el contexto izquierdo, leído de derecha a izquierda (primero L1, luego L2, ...).
- `locale_str="en"` usa las reglas alfabéticas del inglés. Para eso hace falta ICU.

In [ ]:
orden_izq = C.root.add_arrangement_node(
    grouping=None,
    ordering=[
        ("Sort by Token-Level Attribute",
         {"tokens_attribute": "word", "sorting_scope": "left", "offset": 0,
          "case_sensitive": False, "reverse": False, "backwards": False, "locale_str": "en"})
    ])
show_kwic(orden_izq, n=300)

FlexiConc resalta las palabras que usó para ordenar. Recorre la tabla. Vas a notar que el contexto izquierdo varía mucho, **salvo por un patrón muy frecuente**: un posesivo justo antes de *eyes* (*his eyes*, *her eyes*).

---
## 4. Agrupar por la palabra en L1

Para confirmar esa impresión con números, **agrupamos** las líneas según la palabra en L1. El algoritmo *Partition by Ngrams* hace un grupo por cada palabra distinta en las posiciones indicadas, y los ordena de mayor a menor.

Colgamos el nuevo nodo de `orden_izq`. Así, dentro de cada grupo, las líneas siguen ordenadas por la izquierda.

In [ ]:
por_L1 = orden_izq.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [-1], "tokens_attribute": "word", "case_sensitive": False}),
    ordering=[])
show_kwic(por_L1, n=5, n_groups=12)

Cada grupo aparece plegado con su número de líneas. Haz clic en **▶** para desplegarlo. `n=5` muestra sólo cinco líneas por grupo y `n_groups=12` sólo los doce grupos más grandes. Aquí nos interesan las **frecuencias**. Las líneas son sólo ejemplos.

Los grupos también se pueden consultar como datos. `grouping_result` guarda, para cada grupo, su etiqueta y las líneas que contiene. Con eso armamos una tabla de frecuencias:

In [ ]:
import pandas as pd

grupos = por_L1.grouping_result["partitions"]
frec_L1 = pd.DataFrame(
    [(g["label"].strip("(),'"), len(g["line_ids"])) for g in grupos],
    columns=["L1", "lineas"])
frec_L1["porcentaje"] = (100 * frec_L1["lineas"] / C.root.line_count).round(1)
frec_L1.head(15)

### ✅ Ejercicio 1 (resuelto)

1. ¿Qué porcentaje de las líneas tiene un posesivo (*his, her, my, your, our, their, its*) en L1?
2. ¿Qué otras palabras aparecen en L1 con frecuencia? ¿Qué tipo de construcción sugieren?

In [ ]:
posesivos = ["his", "her", "my", "your", "our", "their", "its"]
round(frec_L1[frec_L1["L1"].isin(posesivos)]["porcentaje"].sum(), 1)

**1.** Alrededor del 60 %, sobre todo *his* y *her*. En inglés, las partes del cuerpo casi siempre llevan posesivo (*he shut his eyes*). En español diríamos *cerró los ojos*, con artículo. Lo veremos en el cuaderno 08.

**2.** Aparecen *the* (*the eyes of …*), adjetivos (*bright*, *dark*, *blue*, *black*) y *own* (*his own eyes*). La entrada `s` viene del genitivo sajón: CLiC separa *Florence's* en *Florence* + *'* + *s*, así que *Florence's eyes* también es una construcción posesiva.

Muchas líneas con posesivo son parte de construcciones más largas, como *with his eyes*: *with* es la palabra más frecuente en L2. Para verlo, agrupa por L2 y L1 juntas con `positions: [-2, -1]`.

---
## 5. Filtrar los posesivos y ordenar por otras partes del cuerpo

Ahora nos quedamos sólo con las líneas que tienen posesivo en L1. Es un nodo de **subconjunto**. El algoritmo *Select by Token-Level String Attribute* revisa la palabra en una posición (`offset=-1`) y conserva las líneas en que coincide con alguna de la lista.

In [ ]:
con_posesivo = por_L1.add_subset_node(
    ("Select by Token-Level String Attribute",
     {"search_terms": posesivos, "tokens_attribute": "word", "offset": -1,
      "case_sensitive": False, "regex": False, "negative": False}))
print(con_posesivo.line_count)
show_analysis_tree(C, mark=con_posesivo.id)

`mark` resalta un nodo en el árbol. Nota que el nuevo nodo cuelga de `por_L1`. Por eso hereda el ordenamiento y la agrupación: las líneas siguen agrupadas por posesivo.

In [ ]:
show_kwic(con_posesivo, n=5)

Al leer estas líneas da la impresión de que los ojos aparecen **junto con otras partes del cuerpo**: *his eyes and his hands*, *her eyes upon his face*. Para comprobarlo usamos un algoritmo de ***ranking***.

*KWIC Grouper Ranker* cuenta cuántas palabras de una lista aparecen en una ventana alrededor del nodo y pone primero las líneas con más coincidencias. La ventana va de 20 palabras a la izquierda (`window_start=-20`) a 20 a la derecha (`window_end=20`).

In [ ]:
partes_del_cuerpo = ["face", "hand", "hands", "head", "arm", "arms", "shoulder", "shoulders",
                     "lips", "mouth", "brow", "forehead", "cheek", "cheeks", "chin", "neck",
                     "nose", "hair", "fingers", "knees", "chest", "breast", "waist"]

por_cuerpo = con_posesivo.add_arrangement_node(
    grouping=None,
    ordering=[
        ("KWIC Grouper Ranker",
         {"search_terms": partes_del_cuerpo, "tokens_attribute": "word", "mode": "literal",
          "case_sensitive": False, "window_start": -20, "window_end": 20,
          "include_node": False, "count_types": False})
    ])
show_kwic(por_cuerpo, n=40)

### ✅ Ejercicio 2 (resuelto)

1. ¿Qué partes del cuerpo aparecen más cerca de *eyes*?
2. ¿Qué verbos se repiten en estas líneas? ¿Describen acciones físicas o estados de ánimo?

**1.** Sobre todo *face*, *head* y *hand(s)*: *raised his eyes to her face*, *put his hand before his eyes*. Alrededor de un tercio de las líneas con posesivo menciona otra parte del cuerpo en la ventana. El cuerpo se describe como un conjunto de gestos.

**2.** Dominan los verbos que **dirigen la mirada** (*looked, turned, fixed, raised, cast*) y los que la abren o cierran (*opened, shut, closed*). Son acciones visibles. A través de ellas el narrador sugiere qué siente o piensa el personaje sin decirlo. Ésa es la tesis de Mahlberg (2013) sobre el lenguaje corporal en Dickens.

---
## 6. Ojos en las suspensiones

CLiC divide cada novela en partes:

- `quote`: lo que dicen los personajes, entre comillas.
- `nonquote`: la narración.
- `longsus` (*long suspension*): una interrupción de cinco palabras o más **dentro** de un parlamento.

Una suspensión se ve así:

> "I don't know," **said Mr. Pecksniff, with his eyes fixed on the ceiling,** "that I ever saw a more …"

Mahlberg, Smith y Preston (2013) mostraron que en las suspensiones Dickens concentra el lenguaje corporal: el narrador detiene el diálogo para mostrar qué hace el cuerpo del personaje mientras habla.

Primero veamos cuántas líneas hay en cada parte. *Partition by Metadata Attribute* agrupa por una columna de `C.metadata`.

In [ ]:
por_parte = C.root.add_arrangement_node(
    grouping=("Partition by Metadata Attribute", {"metadata_attribute": "clic_subset"}),
    ordering=[])
show_kwic(por_parte, n=3)

Ahora nos quedamos con las suspensiones largas y las agrupamos por la palabra en R1, para ver qué sigue a *eyes*.

In [ ]:
suspensiones = C.root.add_subset_node(
    ("Select by Metadata Attribute", {"metadata_attribute": "clic_subset", "value": "longsus"}))

susp_R1 = suspensiones.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [1]}),
    ordering=[])
show_kwic(susp_R1, n=5, n_groups=8)

### ✅ Ejercicio 3 (resuelto)

Haz el mismo agrupamiento por R1 con **toda** la concordancia (cuelga el nodo de `C.root`). ¿Cambia el orden de las palabras más frecuentes? ¿Qué te dice eso de las suspensiones?

In [ ]:
todo_R1 = C.root.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [1]}),
    ordering=[])
show_kwic(todo_R1, n=3, n_groups=8)

En la concordancia completa, después de *eyes* vienen sobre todo la coma, el punto, *and* y *were* (*his eyes were …*). En las suspensiones hay dos diferencias:

- **El punto desaparece.** Tiene sentido: una suspensión interrumpe un parlamento que continúa, así que no puede cerrar la oración.
- **Las preposiciones de dirección se duplican.** *To*, *on* y *upon* pasan de 3–4 % a 6–7 %: *with his eyes upon the fire*, *raising her eyes to his*.

En las suspensiones la mirada **se dirige a algo** mientras el personaje habla. Es un patrón de escena, no sólo de vocabulario.

Esta comparación es el núcleo de la lectura distante: no leímos ninguna novela completa, pero encontramos una regularidad del estilo de Dickens que se puede verificar volviendo a los textos.

---
## 7. Agrupación por significado (opcional)

Hasta ahora agrupamos por palabras exactas. También podemos agrupar por **parecido de significado**, con los embeddings del cuaderno 06.

Funciona en dos pasos:

1. **Anotar**: cada línea recibe un vector calculado por un modelo *Sentence Transformers* a partir de una ventana de contexto. Usamos de L5 a R5. Con ventanas más grandes los grupos suelen ser menos claros.
2. **Agrupar**: un algoritmo de *clustering* junta las líneas con vectores parecidos.

La primera vez se descarga el modelo `all-MiniLM-L6-v2` (unos 90 MB). Lo aplicamos a toda la concordancia; en Colab tarda uno o dos minutos.

También existe la versión interactiva: `add_annotation_ui(C)`.

In [ ]:
C.add_annotation(
    ("Annotate with Sentence Transformers",
     {"tokens_attribute": "word", "window_start": -5, "window_end": 5}))
C.metadata.columns

La anotación agrega una columna nueva a `C.metadata` con un vector por línea. Ahora agrupamos las **suspensiones** en 8 grupos con *Flat Clustering by Embeddings*. Dentro de cada grupo, *Random Sort* mezcla las líneas para ver ejemplos variados.

Si la columna de embeddings tiene otro nombre en tu versión, cópialo de la salida anterior.

In [ ]:
columna_emb = [c for c in C.metadata.columns if "embedding" in c.lower()][0]

grupos_sem = suspensiones.add_arrangement_node(
    grouping=("Flat Clustering by Embeddings", {"embeddings_column": columna_emb, "n_partitions": 8}),
    ordering=[("Random Sort", {"seed": 42})])
show_kwic(grupos_sem, n=8)

### Para discutir

- ¿Algún grupo junta líneas con un mismo **gesto** (mirar hacia abajo, levantar la vista, ojos con lágrimas)?
- ¿Hay grupos que parecen formados por la **forma** más que por el significado (por ejemplo, todos con *upon*)?
- El modelo `all-MiniLM-L6-v2` se entrenó con inglés contemporáneo de internet. ¿Qué problemas puede tener con prosa victoriana?

Si un grupo te parece interesante, puedes conservarlo con *Manual Line Selection*. Cambia los números por los de los grupos que elijas y **márcalo** como favorito (`bookmarked`):

In [ ]:
seleccion = grupos_sem.add_subset_node(("Manual Line Selection", {"groups": ["Cluster_0", "Cluster_3"]}))
seleccion.bookmarked = True
show_kwic(seleccion, n=20)

In [ ]:
show_analysis_tree(C, mark=seleccion.id)

---
## 8. Hacia el español

Todo lo que hicimos se apoyó en propiedades del inglés: el posesivo en L1, *with his eyes fixed upon*. Antes de pasar al cuaderno 08, piensa cómo se vería esta concordancia en una novela en español.

### ✅ Ejercicio 4 (resuelto)

Traduce estas frases al español de forma natural y fíjate qué pasa con el posesivo:

1. *He shut his eyes.*
2. *Her eyes filled with tears.*
3. *With his eyes fixed on the fire.*

Si agrupáramos una concordancia de *ojos* por L1, ¿qué palabra esperas que sea la más frecuente?

1. *Cerró los ojos.*
2. *Se le llenaron los ojos de lágrimas.*
3. *Con los ojos fijos en el fuego.*

En español las partes del cuerpo suelen llevar **artículo definido**, no posesivo. El poseedor se recupera del sujeto (*cerró los ojos*) o de un **clítico dativo** (*se **le** llenaron los ojos*). Por eso esperamos que *los* sea lo más frecuente en L1, y el clítico quedará **varias posiciones a la izquierda**, no en L1.

Para estudiar esto necesitamos información que CLiC no tiene: lemas, categorías gramaticales y relaciones sintácticas. En el cuaderno 08 armaremos un corpus propio en español y lo anotaremos con un modelo preentrenado de spaCy.

---
## Referencias

- Evert, S. et al. (2025). *Reading Concordances*. Curso en ESSLLI 2025, Bochum. Materiales: [github.com/reading-concordances/teaching](https://github.com/reading-concordances/teaching).
- Mahlberg, M. (2013). *Corpus Stylistics and Dickens's Fiction*. Routledge.
- Mahlberg, M., Smith, C. y Preston, S. (2013). Phrases in literary contexts: Patterns and distributions of suspensions in Dickens's novels. *International Journal of Corpus Linguistics*, 18(1), 35–56.
- Mahlberg, M., Stockwell, P., de Joode, J., Smith, C. y O'Donnell, M. B. (2016). CLiC Dickens: Novel uses of concordances for the integration of corpus stylistics and cognitive poetics. *Corpora*, 11(3), 433–463.
- Moretti, F. (2013). *Distant Reading*. Verso.
- Sinclair, J. (1991). *Corpus, Concordance, Collocation*. Oxford University Press.